# CascadeAI · Gemma 4 LoRA Fine-Tune (Unsloth)

**Goal:** specialise Gemma 4 E4B on CascadeAI's four agent contracts — Event Detector, Impact Predictor, Dispatcher, Narrative Generator — using **Unsloth** for 2-4× faster training and 60% less VRAM than vanilla Hugging Face PEFT.

This notebook is the **Unsloth Special Track** submission artefact. It is designed to run end-to-end on a Google Colab T4 (free) or A100 in ~15 minutes against the 51 seed examples in [`data/training/cascadeai_finetune.jsonl`](../data/training/cascadeai_finetune.jsonl). For production deployment any humanitarian agency can extend the JSONL with their own playbook and re-train without modifying this notebook.

**Design constraints:**

- Train on Gemma 4 E4B (8B params, ~4.5B effective). Cleanest path to merge → safetensors → Ollama for edge deployment.
- LoRA rank 16, alpha 16, dropout 0, target `all-linear` — matches the SolarHive recipe so judges have a reference.
- BF16 precision, batch size auto-tuned by free GPU VRAM.
- Final export targets: (a) LoRA adapters for hot-swap, (b) merged GGUF for `ollama create`.

**Status (May 2026):** Recipe written and validated against the JSONL schema. We have *not* shipped the merged adapters with the hackathon submission — the production CascadeAI dashboard runs base Gemma 4 because the deterministic cascade graph + native function-calling protocol gives strong-enough grounding without fine-tuning. The recipe is here so that the moment a partner agency wants to specialise on (say) WFP-Kenya playbooks or WHO cluster lead guidance, the path is two commits.

## 1 · Install Unsloth and dependencies

Pinning Unsloth to a recent build that supports Gemma 4's `Gemma4ClippableLinear` and MoE expert layers (standard PEFT cannot handle these).

In [ ]:
!pip install -q --upgrade "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets

## 2 · Load Gemma 4 E4B via FastModel

`FastVisionModel` / `FastModel` is the Unsloth wrapper that knows about Gemma 4's per-layer embeddings and clippable linear layers. We load in 4-bit NF4 to keep memory under 12 GB on a T4.

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-e4b-it",
    max_seq_length=2048,
    dtype=None,                  # auto: bf16 on Ampere+, fp16 otherwise
    load_in_4bit=True,
    # Gemma 4 control tokens for tool calling are recognised by the tokenizer.
)
print(model.config.model_type, tokenizer.chat_template is not None)

## 3 · Attach LoRA adapters

Rank 16 on every linear layer keeps the trainable-parameter count under 1% of model weights (~41M of 8B for E4B). `random_state=3407` mirrors SolarHive's recipe — we want deterministic reproducibility for judge audit.

In [ ]:
model = FastModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules="all-linear",
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
model.print_trainable_parameters()

## 4 · Load the CascadeAI training set

Each row in `cascadeai_finetune.jsonl` is an OpenAI-style `{messages: [system, user, assistant]}` triplet plus a `task` tag (one of `event_detector`, `impact_predictor`, `dispatcher`, `narrative`). The chat template lives in the tokenizer so we just apply it.

The 51 seed examples span Ukraine 2022, Sudan 2023→26, Hormuz 2026, the BEV cascade and the Horn of Africa drought — exactly the scenarios our backtest table validates against.

In [ ]:
from datasets import load_dataset

raw = load_dataset(
    "json",
    data_files="../data/training/cascadeai_finetune.jsonl",
    split="train",
)
print("rows:", len(raw))
print("tasks:", set(raw["task"]))

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = raw.map(format_chat, remove_columns=raw.column_names)
print(dataset[0]["text"][:600])

## 5 · Configure SFTTrainer

Hyperparameters mirror the Unsloth + SolarHive recipe so judges have a like-for-like comparison: `lr=2e-4`, `warmup=5`, `seed=3407`. With 51 examples and effective batch size 8 we want ~3 epochs (≈19 steps); the trainer config below lets the auto-tuner pick the per-device batch given available VRAM.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="cascadeai_lora_e4b",
        report_to="none",
    ),
)

In [ ]:
stats = trainer.train()
print("converged_loss:", stats.training_loss)

## 6 · Quick acceptance test

Sanity-check on one held-out event description. The fine-tuned model should return a JSON object with `node`, `severity`, `region`, `summary`, `secondary_nodes` — and nothing else.

In [ ]:
FastModel.for_inference(model)

test_msgs = [
    {"role": "system", "content": "You are CascadeAI's Event Detector. Classify the crisis into JSON with node, severity, region, summary, secondary_nodes."},
    {"role": "user", "content": "Israeli airstrikes on Yemeni port of Hodeidah; Houthis announce closure of Red Sea to commercial shipping."},
]

inputs = tokenizer.apply_chat_template(test_msgs, return_tensors="pt", add_generation_prompt=True).to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.2)
print(tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True))

## 7 · Save artefacts

Two outputs:

1. **LoRA adapters only** — small (~80 MB), hot-swappable, ideal for fast experimentation.
2. **Merged GGUF** for `ollama create` — packages the fine-tuned weights for the offline / edge deployment path that the demo video features.

In [ ]:
model.save_pretrained("cascadeai_lora_e4b")
tokenizer.save_pretrained("cascadeai_lora_e4b")

model.save_pretrained_merged(
    "cascadeai_e4b_merged",
    tokenizer,
    save_method="merged_16bit",
)

In [ ]:
# Optional: convert to GGUF for Ollama. Requires llama.cpp tooling installed.
model.save_pretrained_gguf(
    "cascadeai_e4b_gguf",
    tokenizer,
    quantization_method="q4_k_m",   # ~5 GB, runs on CPU on Mary's $300 laptop
)

## 8 · Wire the fine-tuned model into CascadeAI

Two paths:

**Ollama (edge path):**

```bash
cd cascadeai_e4b_gguf
cat > Modelfile <<EOF
FROM ./cascadeai_e4b_gguf-q4_k_m.gguf
SYSTEM "You are CascadeAI, a cascading-crisis forecasting agent. Always return valid JSON when an agent contract is requested."
PARAMETER temperature 0.3
PARAMETER num_ctx 8192
EOF
ollama create cascadeai --experimental -f Modelfile
```

Then set `GEMMA_MODEL=cascadeai` in `.env` and the entire CascadeAI pipeline transparently uses the fine-tuned weights.

**Hugging Face hub (cloud path):**

```python
model.push_to_hub_merged(
    "<your-org>/cascadeai-e4b",
    tokenizer,
    save_method="merged_16bit",
    token="<HF_TOKEN>",
)
```

## 9 · Why this is the right path for humanitarian fine-tuning

Three reasons CascadeAI ships *base* Gemma 4 today and fine-tuning as a *recipe*:

1. **Agency-specific playbooks vary.** WFP cluster lead guidance is materially different from MSF field protocols and from the Kenya Red Cross emergency manual. A single fine-tuned model would suppress these differences. Letting each agency clone this notebook with their own JSONL keeps CascadeAI agency-agnostic.
2. **The cascade math is deterministic.** The 11-node BFS graph is the source of truth; Gemma 4 is the language layer. Fine-tuning improves the language layer; it does not change the predictions. Backtest validation already runs at 97.4% (38/39 within range) on base Gemma 4 — the headroom for fine-tuning is in narrative quality, not in numerical accuracy.
3. **Multilingual breadth is the audience-voice differentiator.** Fine-tuning E4B on 51 English-majority examples risks degrading the Swahili / Amharic / Bengali narrative quality CascadeAI's demo specifically highlights. We would need 1,000+ examples per target language before merging into a production model.

This notebook is the path. The data is the seed. Any partner agency can ship their specialised CascadeAI in an afternoon.